# Snippet from Math-Uncertainty-and-Calibration.md


In [ ]:
import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

def calibrate_platt(scores: np.ndarray, labels: np.ndarray) -> callable:
    """Platt scaling: Logistic regression on raw scores"""
    clf = LogisticRegression(fit_intercept=True)
    clf.fit(scores.reshape(-1, 1), labels)
  
    def calib_prob(raw_score):
        z = clf.intercept_ + clf.coef_[0] * raw_score
        return 1 / (1 + np.exp(-z))
    return calib_prob

def conformal_quantiles(residuals: np.ndarray, alpha: float = 0.05) -> float:
    """Conformal prediction: (1-alpha) quantile for coverage"""
    sorted_res = np.sort(residuals)
    q = sorted_res[int((1 - alpha) * len(sorted_res))]
    return q  # e.g., for 95% CI: μ ± q

# Example: Toy Probabilities
raw_probs = np.array([0.9, 0.8, 0.7])  # Overconfident
true_labels = np.array([1, 0, 1])
calib = calibrate_platt(raw_probs, true_labels)
print([calib(p) for p in raw_probs])  # Softer: [0.85, 0.45, 0.75] (approx)

res = np.abs(raw_probs - true_labels)  # Toy residuals
print(conformal_quantiles(res, 0.05))  # ~0.2 for 95% width
